# What one vector per word costs

Averaging two senses into one vector, and the layer weights ELMo learns instead.

From chapter 16, [Contextual Embeddings](https://nlp.jcrlabz.com/book/contextual/), of the course notes.

Source: `book/code/worked_examples/contextual.py`

## What you need

Nothing. This example uses only the Python standard library.

In [ ]:
# Nothing to install: this example uses only the Python standard library.
import sys; print(sys.version.split()[0])

## The script

Everything the example defines, in one cell. Run it and the definitions
are live in the notebook.

In [ ]:
"""Contextual embeddings, worked as Chapter 16 works them.

Four things, all reproducing the chapter's tables:

    the averaging cost   what one vector per word does to 'bank'
    characters in        how a convolution turns spelling into a word vector
    ELMo layer mixing    the softmax weights, and what each layer contributes
    masked prediction    why BERT masks 15 per cent and not 100

    python3 contextual.py           # all four
    python3 contextual.py --bank    # static against contextual
    python3 contextual.py --chars   # the character convolution, step by step
    python3 contextual.py --elmo    # layer weights, three task profiles
    python3 contextual.py --mask    # the masking budget

Install:  nothing, the Python standard library is enough
"""

import argparse
import math

# A toy space on three axes: (money, river, generic). The two senses of 'bank'
# are given separately, as a contextual model would produce them.
SPACE = {
    "bank/money":  (0.90, 0.05, 0.20),
    "bank/river":  (0.05, 0.90, 0.20),
    "deposit":     (0.95, 0.02, 0.15),
    "loan":        (0.92, 0.03, 0.18),
    "vault":       (0.88, 0.01, 0.10),
    "water":       (0.02, 0.93, 0.15),
    "erosion":     (0.01, 0.90, 0.08),
    "flood":       (0.03, 0.95, 0.12),
}


def cosine(u, v):
    d = sum(a * b for a, b in zip(u, v))
    n = math.sqrt(sum(a * a for a in u)) * math.sqrt(sum(b * b for b in v))
    return d / n if n else 0.0


def neighbours(vec, exclude=(), top=4):
    scored = [(w, cosine(vec, v)) for w, v in SPACE.items() if w not in exclude]
    return sorted(scored, key=lambda p: -p[1])[:top]


# ---------------------------------------------------------- characters in

# A character embedding table, size two so every product can be checked by hand.
# Real tables are learned. These are chosen, and their only job is to differ.
CHARS = {"^": (-0.5, 0.2), "$": (0.2, -0.5),
         "a": (0.8, 0.1), "b": (0.1, 0.9), "k": (-0.7, 0.6), "n": (0.6, -0.6),
         "s": (-0.4, -0.8), "i": (0.3, 0.7), "g": (-0.2, -0.3),
         "z": (-0.9, -0.1), "e": (0.7, 0.4), "r": (0.0, -0.9)}

# Four filters. A filter of width w holds one row of two weights per character
# it covers, and a bias.
FILTERS = {"A": ([(0.9, 0.2), (-0.3, 0.6)], -0.10),
           "B": ([(-0.4, 0.5), (0.8, -0.5)], 0.00),
           "C": ([(0.5, 0.3), (0.2, -0.6), (-0.7, 0.1)], 0.05),
           "D": ([(0.1, -0.2), (0.6, 0.4), (-0.2, -0.8)], 0.10)}


def window_score(weights, bias, chunk):
    """One filter on one window: multiply, add up, add the bias."""
    return sum(w * c for row, ch in zip(weights, chunk)
               for w, c in zip(row, CHARS[ch])) + bias


def convolve(word, name):
    """Slide a filter along the padded word. Returns (window, score, relu)."""
    weights, bias = FILTERS[name]
    padded = "^" + word + "$"
    width = len(weights)
    out = []
    for i in range(len(padded) - width + 1):
        chunk = padded[i:i + width]
        score = window_score(weights, bias, chunk)
        out.append((chunk, score, max(0.0, score)))
    return out


def pool(word):
    """The word vector: for every filter, its strongest window."""
    return [max(convolve(word, n), key=lambda r: r[2]) for n in FILTERS]


def show_chars():
    print("\n\nSTEP 2  a word from its characters\n")
    word = "banks"
    padded = "^" + word + "$"
    print("  Pad the word with a start mark ^ and an end mark $, so the edges")
    print("  of a word are visible to the filters. Then look each character up.\n")
    print(f"  {'char':>6}" + "".join(f"{c:>9}" for c in padded))
    print("  " + "-" * (6 + 9 * len(padded)))
    for d, label in enumerate(("dim 1", "dim 2")):
        print(f"  {label:>6}" + "".join(f"{CHARS[c][d]:>+9.2f}" for c in padded))

    print("\n  Four filters. A and B are two characters wide, C and D three.")
    print("  Each row below is the weight for one character position.\n")
    for name, (weights, bias) in FILTERS.items():
        rows = "  ".join(f"({a:+.1f}, {b:+.1f})" for a, b in weights)
        print(f"    {name}  width {len(weights)}   {rows}   bias {bias:+.2f}")

    weights, bias = FILTERS["A"]
    chunk = "nk"
    terms = " ".join(f"{w:+.1f}*{c:+.1f}" for row, ch in zip(weights, chunk)
                     for w, c in zip(row, CHARS[ch]))
    print(f"\n  One window by hand. Filter A on '{chunk}':\n")
    print(f"    {terms} {bias:+.2f} = {window_score(weights, bias, chunk):+.4f}")

    print("\n\n  Slide every filter along the word. The score for each window,")
    print("  then ReLU, which turns a negative score into zero:\n")
    for name in FILTERS:
        rows = convolve(word, name)
        print(f"  filter {name}")
        print("    " + "".join(f"{c:>9}" for c, _, _ in rows))
        print("    " + "".join(f"{s:>+9.4f}" for _, s, _ in rows) + "   score")
        print("    " + "".join(f"{r:>9.4f}" for _, _, r in rows) + "   after ReLU")

    print("\n\n  Max pooling keeps the strongest window of each filter and")
    print("  forgets where it was:\n")
    print(f"  {'filter':>8}{'strongest window':>20}{'value':>9}")
    print("  " + "-" * 37)
    pooled = pool(word)
    for name, (chunk, _, r) in zip(FILTERS, pooled):
        print(f"  {name:>8}{chunk:>20}{r:>9.4f}")
    vec = [r for _, _, r in pooled]
    print(f"\n  The vector for '{word}' is ({', '.join(f'{v:.2f}' for v in vec)}).")
    print("  Seven positions went in and four numbers came out.")

    print("\n\n  Why this is useful: every word gets a vector, and words that")
    print("  share spelling share their strong windows.\n")
    words = ["bank", "banks", "banking", "banker", "zebra"]
    vectors = {w: [r for _, _, r in pool(w)] for w in words}
    print(f"  {'word':<10}{'pooled vector':>32}{'distance to banks':>22}")
    print("  " + "-" * 64)
    for w in words:
        v = vectors[w]
        dist = math.sqrt(sum((a - b) ** 2 for a, b in zip(v, vectors['banks'])))
        print(f"  {w:<10}{'(' + ', '.join(f'{x:.2f}' for x in v) + ')':>32}{dist:>22.3f}")

    print("\n  The length of the word does not matter, since pooling collapses")
    print("  the positions. Which windows won tells you why the words are close:\n")
    print(f"  {'word':<10}" + "".join(f"{n:>8}" for n in FILTERS))
    print("  " + "-" * 42)
    for w in words:
        print(f"  {w:<10}" + "".join(f"{c:>8}" for c, _, _ in pool(w)))
    print("\n  All four bank words win A and C on the same windows, nk and ank,")
    print("  which is most of why their vectors sit close. The differences are")
    print("  small: banking wins B on 'in' at 1.01 against 1.00, and banker wins")
    print("  D on 'ker' at 1.21 against about 0.8. zebra shares no winning window.")
    print("\n  ELMo does this at scale. Its character layer uses 2048 filters,")
    print("  then two highway layers and a projection down to 512 numbers,")
    print("  and every weight in it is learned along with the language model.")


# ------------------------------------------------------------- the averaging

def show_bank():
    money = SPACE["bank/money"]
    river = SPACE["bank/river"]
    static = tuple((a + b) / 2 for a, b in zip(money, river))

    print("\nSTEP 1  what one vector per word costs\n")
    print("  Two senses of 'bank', as a contextual model would give them:\n")
    print(f"    bank (money)  {money}")
    print(f"    bank (river)  {river}")
    print(f"\n  A static embedding has to pick one vector. Trained on a corpus")
    print(f"  with both senses in equal measure, it lands on their average:\n")
    print(f"    bank (static) ({static[0]:.3f}, {static[1]:.3f},"
          f" {static[2]:.3f})")

    print(f"\n  Now ask each of the three for its nearest neighbours.\n")
    for label, vec, ex in [("bank, money sense", money, ("bank/money",)),
                           ("bank, river sense", river, ("bank/river",)),
                           ("bank, static average", static, ())]:
        top = neighbours(vec, exclude=ex + ("bank/money", "bank/river"))
        cells = "   ".join(f"{w} {s:+.3f}" for w, s in top)
        print(f"  {label:<22}{cells}")

    print(f"\n  The two sense vectors give clean, single-topic lists. The")
    print(f"  average gives a list that mixes both and commits to neither.")

    print(f"\n  Measure it. Cosine of the static average against each sense:\n")
    print(f"    to the money sense  {cosine(static, money):.4f}")
    print(f"    to the river sense  {cosine(static, river):.4f}")
    print(f"    the two senses to each other  {cosine(money, river):.4f}")

    print(f"\n  The average sits {cosine(static, money):.2f} from each sense,")
    print(f"  which sounds close until you notice the senses are only")
    print(f"  {cosine(money, river):.2f} from each other. The static vector is")
    print(f"  equally wrong in two directions rather than right in one.")

    print(f"\n  And the damage scales with how balanced the corpus is. A")
    print(f"  corpus that is 90 per cent money sense gives:\n")
    for share in (0.5, 0.7, 0.9, 0.99):
        v = tuple(share * a + (1 - share) * b for a, b in zip(money, river))
        print(f"    {share:>5.0%} money   to money {cosine(v, money):.4f}"
              f"   to river {cosine(v, river):.4f}")
    print(f"\n  So a static vector does not represent a word. It represents")
    print(f"  the word's frequency-weighted mixture in whatever corpus you")
    print(f"  happened to train on.")


# --------------------------------------------------------------- ELMo mixing

def show_elmo():
    print("\n\nSTEP 3  ELMo mixes the layers, and the task picks the mix\n")
    print("  A biLM of L layers gives L+1 representations per token. ELMo")
    print("  combines them with softmax-normalised task weights:\n")
    print("    ELMo_k = gamma * sum_j  s_j * R_kj\n")
    print("  Only s_0..s_L and gamma are learned downstream. The biLM is")
    print("  frozen, which is why ELMo was cheap to adopt.\n")

    layers = ["layer 0, characters", "layer 1, lower biLSTM",
              "layer 2, upper biLSTM"]
    profiles = {
        "part-of-speech tagging": [0.2, 1.8, 0.5],
        "word sense disambiguation": [0.1, 0.6, 2.0],
        "no preference": [1.0, 1.0, 1.0],
    }

    for task, raw in profiles.items():
        m = max(raw)
        e = [math.exp(r - m) for r in raw]
        s = [x / sum(e) for x in e]
        print(f"  {task}")
        print(f"    {'layer':<24}{'raw weight':>12}{'s_j softmax':>14}")
        print("    " + "-" * 50)
        for name, r, w in zip(layers, raw, s):
            print(f"    {name:<24}{r:>12.2f}{w:>14.4f}")
        print(f"    {'':<24}{'':>12}{sum(s):>14.4f}\n")

    print("  The three profiles are the finding of the ELMo paper, stated as")
    print("  weights. Lower layers carry more syntax, upper layers more")
    print("  meaning, and which one a task wants is a fact you can read off")
    print("  the learned s_j rather than a thing you decide in advance.")
    print("\n  That is also the strongest evidence that depth is doing")
    print("  something structured rather than just adding capacity.")


# ------------------------------------------------------------- masking budget

def show_mask():
    print("\n\nSTEP 4  why mask 15 per cent\n")
    print("  BERT trains by hiding tokens and predicting them. The masking")
    print("  rate is a trade, and both ends of it are easy to see.\n")

    n = 512
    print(f"  A sequence of {n} tokens.\n")
    print(f"  {'rate':>7}{'masked':>9}{'visible':>9}"
          f"{'training signal':>18}{'context left':>15}")
    print("  " + "-" * 60)
    for rate in (0.01, 0.15, 0.50, 0.90):
        masked = int(n * rate)
        print(f"  {rate:>6.0%}{masked:>9}{n - masked:>9}"
              f"{masked:>18}{1 - rate:>14.0%}")

    print("\n  At 1 per cent you get 5 predictions from a forward pass that")
    print("  cost you the whole sequence. Training is correct and slow.")
    print("\n  At 90 per cent you get plenty of predictions and almost no")
    print("  context to make them from. The task becomes guessing.")
    print("\n  BERT chose 15 per cent, which gives 76 gradients per sequence")
    print("  with 85 per cent of the sentence still readable. Later work has")
    print("  found higher rates workable on larger models, so the number is")
    print("  a balance point rather than a constant of nature.")

    print("\n\n  THE PRETRAIN AND FINE-TUNE MISMATCH\n")
    print("  The token [MASK] appears in every pretraining batch and never")
    print("  once at fine-tuning time. A model that keyed on it would break")
    print("  the moment it was used for real.")
    print("\n  So of the 15 per cent chosen, BERT does this:\n")
    chosen = 100
    split = [("replaced with [MASK]", 0.80),
             ("replaced with a random token", 0.10),
             ("left unchanged", 0.10)]
    print(f"  {'treatment':<32}{'share':>8}{'per 100 chosen':>17}")
    print("  " + "-" * 58)
    for name, share in split:
        print(f"  {name:<32}{share:>7.0%}{share * chosen:>17.0f}")

    print("\n  The last two rows are the interesting ones. Because a chosen")
    print("  position might be unchanged, the model cannot tell which")
    print("  positions were chosen. So it has to build a usable")
    print("  representation of every token, not just the masked ones.")
    print("\n  The random-token row does something else. It forces the model")
    print("  to notice when a token does not fit its context, which is a")
    print("  skill no amount of [MASK] prediction would teach.")


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    for f in ("bank", "chars", "elmo", "mask"):
        ap.add_argument(f"--{f}", action="store_true")
    a = ap.parse_args()
    picked = a.bank or a.chars or a.elmo or a.mask
    if a.bank or not picked:
        show_bank()
    if a.chars or not picked:
        show_chars()
    if a.elmo or not picked:
        show_elmo()
    if a.mask or not picked:
        show_mask()
    print()

## Running it

The script is a command line tool, so each run below sets the arguments
and calls `main()`. The heading shows the equivalent shell command.

In [ ]:
import sys

def run(*args):
    sys.argv = ["contextual.py", *args]
    main()

### `python3 contextual.py`

The cost of averaging, then the layer weights.

In [ ]:
run()

### `python3 contextual.py --bank`

One vector for two senses sits between both and matches neither.

In [ ]:
run("--bank")

### `python3 contextual.py --chars`

A word built from its characters: padding, four filters, the score of every window, max pooling, and which words end up close together.

In [ ]:
run("--chars")

### `python3 contextual.py --mask`

The masking budget, and what it buys.

In [ ]:
run("--mask")

---

Read the chapter this comes from: [Contextual Embeddings](https://nlp.jcrlabz.com/book/contextual/).